In [ ]:
import json
with open("orders.json", encoding="utf-8") as f:
    orders = json.load(f)
len(orders)    

25

In [32]:
completed_orders = [ order for order in orders if order.get("status") == "completed"]
len(completed_orders)
    

11

In [43]:
countries = list({order.get("country") for order in orders})
len(countries)

5

In [62]:
totales_por_pais = {}

for order in orders:
    if order.get("status") == "completed":
        country = order.get("country")
        total_pedido = order.get("total", 0)
        
        if country:
            totales_por_pais[country] = totales_por_pais.get(country, 0) + total_pedido
for pais, total in totales_por_pais.items():
    print(f"{pais}: {total:.2f}")

DE: 708.94
IT: 378.05
ES: 1006.08
PT: 788.19
FR: 631.11


In [63]:
pedidos_nuevos = 0
pedidos_recurrentes = 0

for order in orders:
    if order.get("is_first_purchase") is True:
        pedidos_nuevos += 1
    else:
        pedidos_recurrentes += 1
print(f"Total de pedidos de clientes nuevos: {pedidos_nuevos}")
print(f"Total de pedidos de clientes recurrentes: {pedidos_recurrentes}")


Total de pedidos de clientes nuevos: 13
Total de pedidos de clientes recurrentes: 12


In [66]:
cantidad_por_producto = {}

for order in orders:
    for item in order.get('items', []):
        nombre_producto = item.get('name')
        cantidad = item.get('quantity', 0)
        if nombre_producto:
            if nombre_producto in cantidad_por_producto:
                cantidad_por_producto[nombre_producto] += cantidad
            else:
                cantidad_por_producto[nombre_producto] = cantidad

if cantidad_por_producto:
    producto_mas_vendido = max(cantidad_por_producto, key=cantidad_por_producto.get)
    cantidad_total = cantidad_por_producto[producto_mas_vendido]

    print(f"{producto_mas_vendido}' {cantidad_total} unidades vendidas ")
else:
    print("No se encontraron productos en los pedidos.")


Pimentón de la Vera' 23 unidades vendidas 


In [67]:
sum_taxes_by_payment_method = {}
count_by_payment_method = {}

for order in orders:
    payment_method = order.get("payment_method")
    tax = order.get("tax", 0) 

    if payment_method: 
        sum_taxes_by_payment_method[payment_method] = sum_taxes_by_payment_method.get(payment_method, 0) + tax
        
        count_by_payment_method[payment_method] = count_by_payment_method.get(payment_method, 0) + 1

print("Promedio de impuestos por método de pago:")

for method, total_tax in sum_taxes_by_payment_method.items():
    count = count_by_payment_method.get(method, 0)
    if count > 0:
        average_tax = total_tax / count
        print(f"{method}: {average_tax:.2f}")
    else:
        print(f"{method}: No hay pedidos para calcular el promedio.")


Promedio de impuestos por método de pago:
card: 96.80
transfer: 109.62
cash: 102.13
paypal: 81.42


In [68]:
refund_reason_counts = {}

for order in orders:
    if order.get("status") == "refunded":
        refund_info = order.get('refund', {})
        
        reason = refund_info.get('refund_reason')
        
        if reason:
            refund_reason_counts[reason] = refund_reason_counts.get(reason, 0) + 1

print("Motivo de reembolso\tCantidad")
for reason, count in refund_reason_counts.items():
    print(f"{reason}\t\t{count}")


Motivo de reembolso	Cantidad
customer_request		2
defective		3
lost_in_transit		4


In [69]:
coupon_analysis = {}

for order in orders:
    coupon_code = order.get("coupon_code")
    discount = order.get("discount", 0)
    
    if coupon_code:
        if coupon_code not in coupon_analysis:
            coupon_analysis[coupon_code] = {'usos': 0, 'descuento_total': 0.0}
        
        coupon_analysis[coupon_code]['usos'] += 1
        coupon_analysis[coupon_code]['descuento_total'] += discount

print("Análisis de cupones usados:")
print(f"{'Cupón':<15} {'Usos':<10} {'Descuento total':<20}")
print("-" * 45)

for code, data in coupon_analysis.items():
    usos = data['usos']
    total_discount = data['descuento_total']
    print(f"{code:<15} {usos:<10} {total_discount:<20.2f}")


Análisis de cupones usados:
Cupón           Usos       Descuento total     
---------------------------------------------
WELCOME10       5          205.26              
SUMMER5         4          20.00               
FREESHIP        5          0.00                


In [70]:
pedidos_vip = [order for order in orders if "vip" in order.get("tags", [])]

productos_vip = {}
for order in pedidos_vip:
    for item in order.get('items', []):
        nombre_producto = item.get('name')
        cantidad = item.get('quantity', 0)
        if nombre_producto:
            productos_vip[nombre_producto] = productos_vip.get(nombre_producto, 0) + cantidad

productos_vip_ordenados = sorted(productos_vip.items(), key=lambda item: item[1], reverse=True)

metodos_pago_vip = {}
for order in pedidos_vip:
    metodo_pago = order.get("payment_method")
    if metodo_pago:
        metodos_pago_vip[metodo_pago] = metodos_pago_vip.get(metodo_pago, 0) + 1

total_gastado_vip = sum(order.get("total", 0) for order in pedidos_vip)
num_pedidos_vip = len(pedidos_vip)
ticket_promedio_vip = total_gastado_vip / num_pedidos_vip if num_pedidos_vip > 0 else 0

print("Análisis de Pedidos VIP:")
print("\nProductos más comprados:")
for producto, cantidad in productos_vip_ordenados:
    print(f"- {producto}: {cantidad}")

print("\nMétodos de pago preferidos:")
for metodo, cantidad in metodos_pago_vip.items():
    print(f"- {metodo}: {cantidad}")

print(f"\nTicket promedio: {ticket_promedio_vip:.2f}")

Análisis de Pedidos VIP:

Productos más comprados:
- Pimentón de la Vera: 12
- Té verde: 8
- Café molido: 7
- Churros mix: 6
- Queso manchego: 5
- Paella kit: 4
- Chocolate 70%: 3

Métodos de pago preferidos:
- paypal: 2
- card: 4
- transfer: 2

Ticket promedio: 698.83
